In [1]:
## KNN Geometric Model

In [2]:
# Imports

In [3]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import wandb

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier


ModuleNotFoundError: No module named 'wandb'

In [4]:
print("[SECTION] W&B authentication")
wandb.login()

[SECTION] W&B authentication


NameError: name 'wandb' is not defined

In [5]:
# Paths y carga de módulos

In [6]:
project_root = Path().resolve().parents[1]  # Ajustado para notebooks
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocessing.one_step import OneStepOptions, preprocess_one_step


ModuleNotFoundError: No module named 'preprocessing'

In [7]:
# Cargar dataset

In [8]:
print("[SECTION] Loading dataset")

df = pd.read_csv(
    project_root / "vehicle-emissions-classification" / "data" / "train.csv"
)

df.head()


[SECTION] Loading dataset


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\rosal\\Documents\\ML\\vehicle-emissions-classification\\data\\train.csv'

In [9]:
# Configurar opciones de preprocesamiento

In [10]:
print("[SECTION] Configuring preprocessing options")

noemp_option = "log"
newexist_option = "A"
createjob_option = "A"
retainedjob_option = "A"
approvaldate_option = "A"
approvalfy_option = "A"
franchise_option = "binary"
urbanrural_option = "onehot"
revlinecr_option = "C"
lowdoc_option = "C"
disbursementgross_option = "A"
local_state = "IL"

options = OneStepOptions(
    noemp_option=noemp_option,
    newexist_option=newexist_option,
    createjob_option=createjob_option,
    retainedjob_option=retainedjob_option,
    approvaldate_option=approvaldate_option,
    approvalfy_option=approvalfy_option,
    franchise_option=franchise_option,
    urbanrural_option=urbanrural_option,
    revlinecr_option=revlinecr_option,
    lowdoc_option=lowdoc_option,
    disbursementgross_option=disbursementgross_option,
    local_state=local_state,
)


[SECTION] Configuring preprocessing options


NameError: name 'OneStepOptions' is not defined

In [11]:
# Preprocesamiento

In [12]:
print("[SECTION] Running preprocessing")

df_processed = preprocess_one_step(df, options=options)

print(f"Rows: {len(df_processed):,}")
print(f"Features: {df_processed.shape[1]}")

df_processed.head()


[SECTION] Running preprocessing


NameError: name 'preprocess_one_step' is not defined

In [13]:
# Train/Holdout Split

In [14]:
print("[SECTION] Building train/holdout split strategy")

target_col = "Accept"
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

print(f"Dataset shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}\n")

X_trainval, X_holdout, y_trainval, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train/Val set size: {X_trainval.shape[0]}")
print(f"Holdout set size: {X_holdout.shape[0]}")
print(f"Train/Val target distribution:\n{y_trainval.value_counts()}\n")
print(f"Holdout target distribution:\n{y_holdout.value_counts()}\n")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval), 1):
    print(f"Fold {fold_idx}: Train={len(train_idx)}, Val={len(val_idx)}")


[SECTION] Building train/holdout split strategy


NameError: name 'df_processed' is not defined

In [15]:
# Inicializar W&B Run

In [16]:
print("[SECTION] Initializing model config and W&B run")

use_scaler = True

run = wandb.init(
    entity="team-data-or-die",
    project="RR Geometric - KNN",
    config={
        "model_name": "KNN",
        "random_state": 42,
        "n_neighbors": 7,
        "metric": "manhattan",
        "weights": "distance",
        "use_scaler": use_scaler,
        "noemp_option": noemp_option,
        "newexist_option": newexist_option,
        "createjob_option": createjob_option,
        "retainedjob_option": retainedjob_option,
        "approvaldate_option": approvaldate_option,
        "approvalfy_option": approvalfy_option,
        "franchise_option": franchise_option,
        "urbanrural_option": urbanrural_option,
        "revlinecr_option": revlinecr_option,
        "lowdoc_option": lowdoc_option,
        "disbursementgross_option": disbursementgross_option,
        "local_state": local_state,
        "n_train_rows": int(X_trainval.shape[0]),
        "n_holdout_rows": int(X_holdout.shape[0]),
        "n_features": int(X_trainval.shape[1]),
    },
)


[SECTION] Initializing model config and W&B run


NameError: name 'wandb' is not defined

In [17]:
# Entrenar KNN

In [18]:
print("[SECTION] Training KNN pipeline")

knn_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler() if use_scaler else "passthrough"),
        ("model", KNeighborsClassifier(
            n_neighbors=7,
            metric="manhattan",
            weights="distance",
        )),
    ]
)

knn_pipeline.fit(X_trainval, y_trainval)


[SECTION] Training KNN pipeline


NameError: name 'Pipeline' is not defined

In [19]:
# Evaluación

In [20]:
print("[SECTION] Running holdout predictions and metric evaluation")

y_pred = knn_pipeline.predict(X_holdout)
y_proba = knn_pipeline.predict_proba(X_holdout)[:, 1]

metrics = {
    "ROC-AUC": roc_auc_score(y_holdout, y_proba),
    "PR-AUC": average_precision_score(y_holdout, y_proba),
    "F1": f1_score(y_holdout, y_pred),
    "Precision": precision_score(y_holdout, y_pred),
    "Recall": recall_score(y_holdout, y_pred),
    "Accuracy": accuracy_score(y_holdout, y_pred),
}

cm = confusion_matrix(y_holdout, y_pred)
report = classification_report(y_holdout, y_pred, output_dict=True)

metrics


[SECTION] Running holdout predictions and metric evaluation


NameError: name 'knn_pipeline' is not defined

In [21]:
# ROC y PR Curves

In [22]:
fpr, tpr, _ = roc_curve(y_holdout, y_proba)
precision, recall, _ = precision_recall_curve(y_holdout, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {metrics['ROC-AUC']:.4f}")
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC Curve")
axes[0].legend()

axes[1].plot(recall, precision, label=f"PR-AUC = {metrics['PR-AUC']:.4f}")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()


NameError: name 'roc_curve' is not defined

In [23]:
# Logging en W&B

In [24]:
wandb.log(metrics)

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_holdout.tolist(),
        preds=y_pred.tolist(),
        class_names=["Reject", "Accept"],
    ),
    "roc_pr_curves": wandb.Image(fig),
})

run.finish()


NameError: name 'wandb' is not defined